# atari_50m_endpoint: small memory setting

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from analysis.curves import (
    episode_average,
    return_curve,
    subsample,
)
from analysis.stats import mean_ci, min_max_normalize
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each agent's stored runs per game.
AGENTS = {
    "Large recency (1M)": "ddqn_atari",
    "Small recency (20k)": "ddqn_small_atari",
    "Small recency (20k) 10-step": "ddqn_small_nstep_atari",
    "Endpoint replay (20k)": "endpoint_small_atari",
}
component = EXPERIMENT.component("ddqn_atari")
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP

results = {}
for agent, name in AGENTS.items():
    for game in GAMES:
        runs = load_results(EXPERIMENT, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs

In [ ]:
# Compute each agent's mean return per game with a 95% bootstrap CI, and each
# run's episode-average return for the bar plot.
curves = {}
episode_scores = {}
for (agent, game), runs in results.items():
    curve = return_curve(runs.reward, runs.done)
    timesteps, returns = subsample(curve)
    curves[agent, game] = (timesteps * FRAMESKIP, mean_ci(returns), len(returns))
    episode_scores[agent, game] = episode_average(runs.reward, runs.done)

In [ ]:
# Line styles and the panel drawing shared by the grid.
LINE_WIDTH = 1.75
LINES = {
    "Large recency (1M)": {"color": "black"},
    "Small recency (20k)": {"color": "tab:blue"},
    "Small recency (20k) 10-step": {"color": "tab:purple"},
    "Endpoint replay (20k)": {"color": "tab:red"},
}
BAND_ALPHA = 0.2


def title(game):
    return game.replace("_", " ").title()


def draw(ax, agent, game):
    frames, (mean, low, high), _ = curves[agent, game]
    line = {"linewidth": LINE_WIDTH, **LINES[agent]}
    ax.fill_between(frames, low, high, color=line["color"], alpha=BAND_ALPHA)
    ax.plot(frames, mean, label=agent, **line)

In [ ]:
# Style for the all-games grid, laid out as in atari_50m_deterministic.
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP
GRID_STYLE = {
    "figure.figsize": (16, 9.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (3, 4)
PANELS = {  # (column, row) of each game's panel
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.06, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.055, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {
    "loc": "upper center",
    "bbox_to_anchor": (0.5, 0.035),
    "ncol": 4,
    "handlelength": 1,
    "handletextpad": 0.4,
    "columnspacing": 1.2,
}
GRID_LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 12}

In [ ]:
# Plot every game in one grid. Seed counts can differ per game, so the shared
# legend names only the agents.
grid = None
if curves:
    with plt.rc_context(GRID_STYLE):
        grid, axes = plt.subplots(*GRID_SHAPE)
        shown = set()
        for game, (column, row) in PANELS.items():
            ax = axes[row, column]
            ax.set_box_aspect(PANEL_ASPECT)
            drawn = [agent for agent in AGENTS if (agent, game) in curves]
            if not drawn:
                ax.axis("off")
                continue
            for agent in drawn:
                draw(ax, agent, game)
            ax.set_title(title(game))
            ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
            shown.update(drawn)
        grid.tight_layout(**GRID_LAYOUT)
        grid.supxlabel(**GRID_XLABEL)
        grid.supylabel(**GRID_YLABEL)
        labels = [agent for agent in AGENTS if agent in shown]
        handles = [
            Line2D([], [], color=LINES[agent]["color"], **GRID_LEGEND_MARKER)
            for agent in labels
        ]
        grid.legend(handles, labels, **GRID_LEGEND)
    plt.show()

In [ ]:
# Style for the normalized-score bar plot.
BAR_ORDER = [
    "Endpoint replay (20k)",
    "Large recency (1M)",
    "Small recency (20k)",
    "Small recency (20k) 10-step",
]
BAR_STYLE = {
    "figure.figsize": (7, 7),
    "font.family": "Times New Roman",
    "font.size": 32,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 32,
    "axes.labelsize": 32,
    "ytick.labelsize": 32,
}
# Bars are named inside, reading up from their base; the last bar is too short
# for that, so its name sits on top of it.
BAR_INSIDE_LABELS = ["Endpoint", "Large", "Small"]
BAR_INSIDE_LABEL = {"rotation": 90, "ha": "center", "va": "bottom", "color": "white"}
BAR_TOP_LABEL = "Small\n10-step"
BAR_LABEL_GAP = 0.01
BAR_WIDTH = 0.6  # a quarter thinner than the default 0.8
# Memory relative to the 1M large-recency buffer.
BAR_TITLE = "Atari 50x Smaller"
BAR_YLABEL = "Normalized Average Return"
BAR_YLABEL_PAD = 0
BAR_YTICKS = [0, 0.8]
BAR_ASPECT = 1

In [ ]:
# Normalize each game's runs onto [0, 1] with bounds shared by the four agents,
# then average over games and seeds.
def bar_plot(scores):
    complete = [g for g in GAMES if all((agent, g) in scores for agent in AGENTS)]
    if not complete:
        return None
    normalized = {}
    for game in complete:
        stacks = min_max_normalize([scores[agent, game] for agent in BAR_ORDER])
        for agent, stack in zip(BAR_ORDER, stacks, strict=True):
            normalized[agent, game] = stack
    with plt.rc_context(BAR_STYLE):
        fig, ax = plt.subplots()
        for position, agent in enumerate(BAR_ORDER):
            mean = np.mean([normalized[agent, g] for g in complete])
            color = LINES[agent]["color"]
            ax.bar(position, mean, width=BAR_WIDTH, color=color)
        for position, label in enumerate(BAR_INSIDE_LABELS):
            ax.text(position, BAR_LABEL_GAP, label, **BAR_INSIDE_LABEL)
        top = (len(BAR_ORDER) - 1, mean + BAR_LABEL_GAP)
        ax.text(*top, BAR_TOP_LABEL, ha="center", va="bottom")
        ax.set_title(BAR_TITLE)
        ax.set_xticks([])
        ax.set_box_aspect(BAR_ASPECT)
        ax.set_ylim(BAR_YTICKS[0], BAR_YTICKS[-1])
        ax.set_yticks(BAR_YTICKS)
        ax.set_ylabel(BAR_YLABEL, labelpad=BAR_YLABEL_PAD)
        fig.tight_layout()
    return fig


bar_figure = bar_plot(episode_scores)
plt.show()

In [ ]:
# Save the grid and the bar plot as a PDF and PNG in the plots dir.
PLOTS_DIR.mkdir(exist_ok=True)
if bar_figure is not None:
    for suffix in ("pdf", "png"):
        name = f"bars_episode_small.{suffix}"
        bar_figure.savefig(PLOTS_DIR / name, bbox_inches="tight")
if grid is not None:
    for suffix in ("pdf", "png"):
        grid.savefig(PLOTS_DIR / f"grid_small.{suffix}", bbox_inches="tight")
print(f"saved the small setting's plots to {PLOTS_DIR}")